<a href="https://colab.research.google.com/github/MamidiPravallikaReddy/DL_LAB/blob/main/DL_week5.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [7]:
import tensorflow as tf
import numpy as np
import scipy.io
from tensorflow.keras import layers, models, regularizers
from tensorflow.keras.utils import to_categorical

!wget http://ufldl.stanford.edu/housenumbers/train_32x32.mat
!wget http://ufldl.stanford.edu/housenumbers/test_32x32.mat

train = scipy.io.loadmat('train_32x32.mat')
test = scipy.io.loadmat('test_32x32.mat')

x_train = np.transpose(train['X'], (3,0,1,2))
y_train = train['y']

x_test = np.transpose(test['X'], (3,0,1,2))
y_test = test['y']

y_train[y_train == 10] = 0
y_test[y_test == 10] = 0

x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

x_train_flat = x_train.reshape(-1, 3072)
x_test_flat = x_test.reshape(-1, 3072)

y_train = to_categorical(y_train, 10)
y_test = to_categorical(y_test, 10)

--2026-04-14 06:25:56--  http://ufldl.stanford.edu/housenumbers/train_32x32.mat
Resolving ufldl.stanford.edu (ufldl.stanford.edu)... 171.64.68.10
Connecting to ufldl.stanford.edu (ufldl.stanford.edu)|171.64.68.10|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 182040794 (174M) [text/plain]
Saving to: ‘train_32x32.mat.3’

train_32x32.mat.3   100%[===================>] 173.61M  6.45MB/s    in 25s     

2026-04-14 06:26:22 (6.82 MB/s) - ‘train_32x32.mat.3’ saved [182040794/182040794]

--2026-04-14 06:26:22--  http://ufldl.stanford.edu/housenumbers/test_32x32.mat
Resolving ufldl.stanford.edu (ufldl.stanford.edu)... 171.64.68.10
Connecting to ufldl.stanford.edu (ufldl.stanford.edu)|171.64.68.10|:80... connected.
HTTP request sent, awaiting response... 200 OK
Length: 64275384 (61M) [text/plain]
Saving to: ‘test_32x32.mat.3’

test_32x32.mat.3    100%[===================>]  61.30M  8.54MB/s    in 9.9s    

2026-04-14 06:26:32 (6.18 MB/s) - ‘test_32x32.mat.3’ saved [642

**L2 Regularization**

In [8]:
model = models.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(512, activation='relu',
                 kernel_regularizer=regularizers.l2(0.0005)),
    layers.Dense(256, activation='relu',
                 kernel_regularizer=regularizers.l2(0.0005)),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(x_train_flat[:30000], y_train[:30000],
          epochs=15, batch_size=128, validation_split=0.2)

loss, acc = model.evaluate(x_test_flat[:5000], y_test[:5000])
print("L2 Accuracy:", acc)

Epoch 1/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 5s 16ms/step - accuracy: 0.1780 - loss: 2.6112 - val_accuracy: 0.1892 - val_loss: 2.4518
Epoch 2/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.1918 - loss: 2.3884 - val_accuracy: 0.1910 - val_loss: 2.2923
Epoch 3/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 9ms/step - accuracy: 0.3456 - loss: 2.0001 - val_accuracy: 0.4625 - val_loss: 1.7034
Epoch 4/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.5084 - loss: 1.5746 - val_accuracy: 0.5495 - val_loss: 1.4372
Epoch 5/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.5732 - loss: 1.3990 - val_accuracy: 0.5942 - val_loss: 1.3436
Epoch 6/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6138 - loss: 1.2834 - val_accuracy: 0.6355 - val_loss: 1.2547
Epoch 7/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6398 - loss: 1.2225 - val_accuracy: 0.5797 - val_loss: 1.3551
Epoch 8/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.6507 - loss: 1.1870 - val_accuracy: 0

**Dataset Augmentation**

In [10]:
from tensorflow.keras.preprocessing.image import ImageDataGenerator
from tensorflow.keras import layers, models

datagen = ImageDataGenerator(
    rotation_range=10,
    zoom_range=0.1,
    width_shift_range=0.1,
    height_shift_range=0.1
)

model = models.Sequential([
    layers.Input(shape=(32,32,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(10,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    datagen.flow(x_train[:30000], y_train[:30000], batch_size=128),
    epochs=15
)

loss, acc = model.evaluate(x_test[:5000], y_test[:5000])
print("Augmentation Accuracy:", acc)

Epoch 1/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 27s 97ms/step - accuracy: 0.3271 - loss: 1.9410
Epoch 2/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 91ms/step - accuracy: 0.6372 - loss: 1.1662
Epoch 3/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 85ms/step - accuracy: 0.7018 - loss: 0.9897
Epoch 4/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 22s 92ms/step - accuracy: 0.7241 - loss: 0.9127
Epoch 5/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 85ms/step - accuracy: 0.7415 - loss: 0.8635
Epoch 6/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 89ms/step - accuracy: 0.7560 - loss: 0.8090
Epoch 7/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 87ms/step - accuracy: 0.7706 - loss: 0.7699
Epoch 8/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 88ms/step - accuracy: 0.7794 - loss: 0.7357
Epoch 9/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 21s 90ms/step - accuracy: 0.7901 - loss: 0.6957
Epoch 10/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 41s 90ms/step - accuracy: 0.8034 - loss: 0.6649
Epoch 11/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 20s 86ms/step - accuracy: 0.8077 - loss: 0.6426
Epoch 12/15
235/235 ━━━━━━━━━━

**Parameter Sharing & Tying**

In [11]:
model = models.Sequential([
    layers.Input(shape=(32,32,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(10,activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(x_train[:30000], y_train[:30000],
          epochs=15, batch_size=128)

loss, acc = model.evaluate(x_test[:5000], y_test[:5000])
print("CNN Accuracy:", acc)

Epoch 1/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 7s 11ms/step - accuracy: 0.4692 - loss: 1.5923
Epoch 2/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8007 - loss: 0.6971
Epoch 3/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8369 - loss: 0.5810
Epoch 4/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8534 - loss: 0.5274
Epoch 5/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8631 - loss: 0.4841
Epoch 6/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8729 - loss: 0.4510
Epoch 7/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8812 - loss: 0.4220
Epoch 8/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8900 - loss: 0.3891
Epoch 9/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8965 - loss: 0.3644
Epoch 10/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9017 - loss: 0.3436
Epoch 11/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.9080 - loss: 0.3210
Epoch 12/15
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/ste

**Adding Noise**

In [13]:
noise_factor = 0.1

x_train_noisy = x_train + noise_factor * np.random.normal(size=x_train.shape)
x_train_noisy = np.clip(x_train_noisy, 0., 1.)

model = models.Sequential([
    layers.Input(shape=(32,32,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(),

    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dense(10,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    x_train_noisy[:30000], y_train[:30000],
    epochs=15,
    batch_size=128,
    validation_split=0.2
)

loss, acc = model.evaluate(x_test[:5000], y_test[:5000])
print("Noise Model Accuracy:", acc)

Epoch 1/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 8s 23ms/step - accuracy: 0.2552 - loss: 2.1078 - val_accuracy: 0.5320 - val_loss: 1.5553
Epoch 2/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6748 - loss: 1.0557 - val_accuracy: 0.7473 - val_loss: 0.8421
Epoch 3/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.7689 - loss: 0.7640 - val_accuracy: 0.7742 - val_loss: 0.7637
Epoch 4/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7958 - loss: 0.6807 - val_accuracy: 0.7903 - val_loss: 0.7251
Epoch 5/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8089 - loss: 0.6374 - val_accuracy: 0.8040 - val_loss: 0.6813
Epoch 6/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8207 - loss: 0.5981 - val_accuracy: 0.8030 - val_loss: 0.6727
Epoch 7/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8311 - loss: 0.5695 - val_accuracy: 0.8100 - val_loss: 0.6549
Epoch 8/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8384 - loss: 0.5371 - val_accuracy: 0

**Early Stopping**

In [14]:
from tensorflow.keras.callbacks import EarlyStopping

early_stop = EarlyStopping(monitor='val_loss', patience=3)

model = models.Sequential([
    layers.Input(shape=(3072,)),
    layers.Dense(256, activation='relu'),
    layers.Dense(10, activation='softmax')
])

model.compile(optimizer='adam',
              loss='categorical_crossentropy',
              metrics=['accuracy'])

model.fit(x_train_flat[:30000], y_train[:30000],
          epochs=20,
          batch_size=128,
          validation_split=0.2,
          callbacks=[early_stop])

loss, acc = model.evaluate(x_test_flat[:5000], y_test[:5000])
print("EarlyStopping Accuracy:", acc)

Epoch 1/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 4s 13ms/step - accuracy: 0.1714 - loss: 2.3541 - val_accuracy: 0.1883 - val_loss: 2.2692
Epoch 2/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.1956 - loss: 2.2315 - val_accuracy: 0.2088 - val_loss: 2.2162
Epoch 3/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2083 - loss: 2.2086 - val_accuracy: 0.1985 - val_loss: 2.2011
Epoch 4/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.2217 - loss: 2.1552 - val_accuracy: 0.2610 - val_loss: 2.1083
Epoch 5/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.2827 - loss: 2.0235 - val_accuracy: 0.3415 - val_loss: 1.9568
Epoch 6/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.3768 - loss: 1.8330 - val_accuracy: 0.4105 - val_loss: 1.7520
Epoch 7/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.4343 - loss: 1.6911 - val_accuracy: 0.4452 - val_loss: 1.6449
Epoch 8/20
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 4ms/step - accuracy: 0.4630 - loss: 1.5946 - val_accuracy: 0

**Ensemble Method**

In [17]:
def build_model():
    model = models.Sequential([
        layers.Input(shape=(32,32,3)),

        layers.Conv2D(32,(3,3),activation='relu'),
        layers.MaxPooling2D(),

        layers.Conv2D(64,(3,3),activation='relu'),
        layers.MaxPooling2D(),

        layers.Flatten(),
        layers.Dense(128,activation='relu'),
        layers.Dense(10,activation='softmax')
    ])

    model.compile(
        optimizer='adam',
        loss='categorical_crossentropy',
        metrics=['accuracy']
    )
    return model

num_models = 3
models_list = []

for i in range(num_models):
    print(f"\nTraining Model {i+1}...")

    model = build_model()

    model.fit(
        x_train[:30000], y_train[:30000],
        epochs=10,
        batch_size=128,
        verbose=1
    )

    models_list.append(model)

predictions = []

for model in models_list:
    pred = model.predict(x_test[:5000])
    predictions.append(pred)

avg_predictions = np.mean(predictions, axis=0)

final_pred = np.argmax(avg_predictions, axis=1)
true_labels = np.argmax(y_test[:5000], axis=1)

accuracy = np.mean(final_pred == true_labels)

print("\nEnsemble Accuracy:", accuracy)


Training Model 1...
Epoch 1/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 14ms/step - accuracy: 0.4994 - loss: 1.5162
Epoch 2/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7953 - loss: 0.7178
Epoch 3/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 7ms/step - accuracy: 0.8318 - loss: 0.5962
Epoch 4/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 2s 6ms/step - accuracy: 0.8516 - loss: 0.5338
Epoch 5/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8636 - loss: 0.4887
Epoch 6/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8736 - loss: 0.4532
Epoch 7/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8813 - loss: 0.4180
Epoch 8/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8884 - loss: 0.3909
Epoch 9/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 5ms/step - accuracy: 0.8974 - loss: 0.3650
Epoch 10/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.9014 - loss: 0.3442

Training Model 2...
Epoch 1/10
235/235 ━━━━━━━━━━━━━━━━━━━━ 6s 11ms/step - accuracy: 0.4788 - loss: 1.5740
Epoch 2/1

**Dropout**

In [18]:
model = models.Sequential([
    layers.Input(shape=(32,32,3)),

    layers.Conv2D(32,(3,3),activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Conv2D(64,(3,3),activation='relu'),
    layers.MaxPooling2D(),
    layers.Dropout(0.25),

    layers.Flatten(),
    layers.Dense(128,activation='relu'),
    layers.Dropout(0.4),

    layers.Dense(10,activation='softmax')
])

model.compile(
    optimizer='adam',
    loss='categorical_crossentropy',
    metrics=['accuracy']
)

model.fit(
    x_train[:30000], y_train[:30000],
    epochs=15,
    batch_size=128,
    validation_split=0.2
)

loss, acc = model.evaluate(x_test[:5000], y_test[:5000])
print("Dropout Accuracy:", acc)

Epoch 1/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 15s 47ms/step - accuracy: 0.2272 - loss: 2.1606 - val_accuracy: 0.4387 - val_loss: 1.7027
Epoch 2/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.6025 - loss: 1.2424 - val_accuracy: 0.7907 - val_loss: 0.7859
Epoch 3/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.7425 - loss: 0.8348 - val_accuracy: 0.8298 - val_loss: 0.6117
Epoch 4/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.7815 - loss: 0.7181 - val_accuracy: 0.8377 - val_loss: 0.5683
Epoch 5/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 6ms/step - accuracy: 0.8001 - loss: 0.6562 - val_accuracy: 0.8517 - val_loss: 0.5200
Epoch 6/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8150 - loss: 0.6080 - val_accuracy: 0.8585 - val_loss: 0.4835
Epoch 7/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 1s 7ms/step - accuracy: 0.8239 - loss: 0.5755 - val_accuracy: 0.8567 - val_loss: 0.4907
Epoch 8/15
188/188 ━━━━━━━━━━━━━━━━━━━━ 2s 8ms/step - accuracy: 0.8343 - loss: 0.5411 - val_accuracy: 